# Fairness Audit — Disparate Impact Across Groups

**Question:** *Are any groups approved at materially different rates than the overall portfolio?*

We compute approval rates by loan grade and employment length, flag groups whose approval rate deviates more than **20%** from the overall rate (a common EEOC-style rule of thumb), and also report the four-fifths ratio.

**Caveat:** grade is itself an underwriting input, so disparity across grades is expected. We surface it anyway to put a number on it.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src import data_loader, fairness, metrics as M, model as Mdl

trained = Mdl.load_model()
df = data_loader.load_clean()
df['pd'] = trained.predict_pd(df)

# Approve iff expected profit > 0 (per-loan break-even).
df['exp_profit'] = M.expected_profit(df['pd'], df['loan_amnt'],
                                     df['int_rate'], df['term_months'])
df['approve'] = (df['exp_profit'] > 0).astype(int)
df['approve'].mean()

## By loan grade

In [ ]:
by_grade = fairness.audit_by_grade(df)
by_grade

In [ ]:
fairness.summarise(by_grade, 'grade_letter')

## By employment length

In [ ]:
by_emp = fairness.audit_by_emp_length(df)
by_emp

In [ ]:
fairness.summarise(by_emp, 'emp_length_raw')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, audit_df, col, title in [
    (axes[0], by_grade, 'grade_letter', 'Approval rate by grade'),
    (axes[1], by_emp, 'emp_length_raw', 'Approval rate by emp length'),
]:
    colors = ['#e74c3c' if f else '#2980b9' for f in audit_df['flag_20pct']]
    ax.bar(audit_df[col].astype(str), audit_df['approval_rate'], color=colors)
    ax.axhline(audit_df['overall_approval_rate'].iloc[0],
               color='black', linestyle='--', label='Overall')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Approval rate')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/fairness_audit.png', dpi=140, bbox_inches='tight')
plt.show()